## Εργασία 3 ##

Καλωσήρθατε στην τρίτη σας εργασία. Η εργασία αυτή έχει σκοπό να σας βοηθήσει να εμπεδώσετε τα σύνολα μοντέλων.

Στην εργασία αυτή θα πρέπει να συμπληρώσετε κώδικα Python 3 στα σημεία που αναφέρουν # YOUR CODE HERE. Μην τροποποιείτε τον κώδικα που βρίσκεται εκτός αυτών των περιοχών.

Πρωτού παραδόσετε την εργασία σας σιγουρευτείτε ότι ο κώδικας σε όλα τα κελιά τρέχει σωστά. Για το σκοπό αυτό επιλέξτε από το μενού **Χρόνος εκτέλεσης (runtime) -> Επανεκίνηση περιόδου λειτουργίας και εκτέλεση όλων**.

Συμπληρώστε το όνομα (**NAME**) και το **AEM** σας παρακάτω **εντός των διπλών εισαγωγικών**:

In [83]:
NAME = "Βλαχάκης Θεόδωρος"
AEM = "10476"

**1 (2 μονάδες)** Σε αυτήν την εργασία θα δουλέψουμε με ένα σύνολο δεδομένων δυαδικής ταξινόμησης, από το δημόσιο αποθετήριο δεδομένων και πειραμάτων μηχανικής μάθησης [openml.org](https://www.openml.org/). Μπορείτε να δείτε πως φορτώνουμε σύνολα δεδομένων από το openml.org μέσα από την sklearn μελετώντας την αντίστοιχη [τεκμηρίωση](https://scikit-learn.org/stable/datasets/loading_other_datasets.html#downloading-datasets-from-the-openml-org-repository). Συγκεκριμένα θέλουμε να δουλέψουμε με την έκδοση 1 του Pima Indians Diabetes Database, το οποίο αφορά τη διάγνωση διαβήτη. Χωρίστε το σύνολο δεδομένων σε δεδομένα εκπαίδευσης (X_train, y_train) και ελέγχου (X_test, y_test) σε ποσοστό 70% και 30% αντίστοιχα με τη συνάρτηση train_test_split (τιμή για random_state βάλτε 0).

In [84]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split

data = fetch_openml(name='diabetes', version=1)

# print(data.details['id'])
# print(data['data'].shape)
# print(data.target.shape)
#print(data.feature_names)
X = data['data']
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
#print(y_train)

In [85]:
"""Τεστ ορθής ανάγνωσης και διαχωρισμού του συνόλου δεδομένων"""
assert X_train['mass'][0] == 33.6
assert X_test['pedi'][34] == 0.512

**2 (4 μονάδες)** Υλοποιήστε μια ντετερμινιστική εκδοχή της μεθόδου των τυχαίων υποχώρων, η οποία χτίζει τόσα μοντέλα όσες και οι μεταβλητές εισόδου, κάθε ένα από τα οποία αγνοεί και μία διαφορετική μεταβλητή εισόδου. Π.χ. το πρώτο μοντέλο αγνοεί την πρώτη, το δεύτερο αγνοεί τη δεύτερη κτλ. Χρησιμοποιήστε τη συνάρτηση clone από το sklearn.base για να δημιουργήστε αντίγραφο του βασικού μοντέλου σε κάθε επανάληψη. 

In [86]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

class RandomSubspaceDet:
    def __init__(self, estimator=DecisionTreeClassifier()):
        """
        The function __init__ is the initialization function of the class RandomSubspaceDet
        
        """

        # Initialize the base estimator model and the number of learning
        # models/estimators that will comprise the model set, constructed by the Random
        # Subspace method, to 0. Also, initialize the model set to an empty set.
        self.base_estimator = estimator
        self.n_estimators = 0
        self.estimators_set = []

    def fit(self, X_train, y_train):
        """
        The fit function is the implementation of the fitting procedure of the models/estimators
        produced by the following deterministic version of the Random Subspace method. According
        to it, the number of the models produced is equal to the number of the input variables in
        the training set and each of these models is trained on a variation of this training set, 
        which is obtained by ignoring a different input variable each time.
        This function does not return anything.

        """
        self.n_estimators = X_train.shape[1]
        estimator_name_base_str = 'estimator_'

        for excluded_feature in range(self.n_estimators):
            current_estimator = clone(self.base_estimator)
            current_X_train = np.delete(X_train, excluded_feature, axis=1)

            current_estimator.fit(current_X_train, y_train)

            # Update the model set with the new estimator
            self.estimators_set.append((estimator_name_base_str + f"{excluded_feature+1}", current_estimator, excluded_feature))

    def predict(self, X):
        """
        The predict function makes predictions for the instancees in a data set X by combining the
        predictions of the individual estimators in the model set for those instances in X.
        The function returns the class into which it classifies -considering the logic of "soft" or
        "hard" voting across all the predictions of the individual estimators in model set for each
        instance in X- each instance of X, in the case of a classification problem, and the value
        (real number) it predicts -considering an average of the predictions of the individual estimators
        in model set for each instance in X- for the target variable of each instance in X, in the case
        of a regression problem.

        """
        
        # First of all, check if the problem is a classification or a regression problem
        problem_type = 'regression'
        if (hasattr(self.estimators_set[0][1], 'classes_') or hasattr(self.estimators_set[0][1], 'predict_proba')):
            problem_type = 'classification'
        
        # Since recognizing the type of problem, calculate the total prediction given by the set
        # of models built and trained, on some training set, using the Random Subspace method.
        #individual_predictions = np.array((X.shape[0], len(self.estimators_set)))

        if (problem_type == 'regression'):
            # Ensemble prediction aggregation by averaging the predictions of the individual
            # estimators on the set X.
            # Define a 2D array called individual predictions whose each column stores the
            # predictions, made by the corresponding estimator model in the model set, for each
            # instance in X.
            # Then calculate the total predictions for each instance in X, averaging the
            # individual predictions of every estimator in model set for this instance in X.

            individual_predictions = np.array(
                [estimator.predict(np.delete(X, excluded_feature, axis=1)) for (_, estimator, excluded_feature) in self.estimators_set]).T
            total_predictions = np.average(individual_predictions, axis=1).reshape(-1)
        
        else:
            # This means that the problem is a classification problem

            # Check if the estimators in model set can calculate/predict the probability that
            # an instance in X belongs to a class or they directly predict the class to which
            # this instance will be classified. If they can predict probabilities use them in
            # a "soft voting" logic to make the final predictions. Instead use a "hard voting"
            # logic.
            if(hasattr(self.estimators_set[0][1], 'predict_proba')):
                # Define a 3D array called individual predictions whose the first dimension is
                # equal to the number of estimators in model set. The other two dimensions
                # construct a 2D array which is the result of predict_proba function for the
                # corresponding estimator (i.e it is a 2D array whose each row refers to an
                # instance in X and includes the probabilities for all the classes in the classification
                # problem).
                
                individual_predictions = np.array([
                    estimator.predict_proba(np.delete(X, excluded_feature, axis=1)) for (_, estimator, excluded_feature) in self.estimators_set
                    ])
                total_predictions_proba = np.average(individual_predictions, axis=0)
                total_predictions = np.argmax(total_predictions_proba, axis=1).reshape(-1)

                # Check if the final predictions must be the name of the class for each instance in X
                if(hasattr(self.estimators_set[0][1], 'classes_') and type(self.estimators_set[0][1].predict(np.delete(X, 0, axis=1))[0]) == str):
                    classes = self.estimators_set[0][1].classes_
                    total_predictions_classes = classes[total_predictions]
                    
                    # Refactor the np array total_predictions
                    total_predictions = np.array(total_predictions_classes)

            else:
                # Define a 2D array called individual predictions whose each column stores the
                # predictions, made by the corresponding estimator model in the model set, for 
                # the class of each instance in X.
                # Then calculate the total predictions for the class of each instance in X, considering
                # a "hard voting" logic across all the individual predictions given by estimators
                # in model set.
                individual_predictions = np.array(
                    [estimator.predict(np.delete(X, excluded_feature, axis=1)) for (_, estimator, excluded_feature) in self.estimators_set]).T
                
                # Check if the predictions of the individual estimators are the names of the classes
                # for each instance in X or not. In the first case, the final predictions must also
                # be the name of the class for each instance in X.
                if(type(individual_predictions[0][0]) == str):
                    classes = self.estimators_set[0][1].classes_
                    
                    encoding = {cls: index for (index, cls) in enumerate(classes)}

                    individual_predictions_encoded = np.vectorize(encoding.get)(individual_predictions)

                    total_predictions_encoded = np.zeros((X.shape[0]),)

                    for i in range(total_predictions_encoded.shape[0]):
                        counting_classes = np.bincount(individual_predictions_encoded[i,:])
                        total_predictions_encoded[i] = np.argmax(counting_classes)
                    
                    # Decoding procedure
                    total_predictions = np.array(classes[total_predictions])

                else:
                    total_predictions = np.zeros(X.shape[0],)
                    for i in range(total_predictions.shape[0]):
                        counting_classes = np.bincount(individual_predictions[i,:])
                        total_predictions[i] = np.argmax(counting_classes)

        return total_predictions

In [87]:
"""Τεστ ορθής υλοποίησης RandomSubspaceDet"""
from sklearn.metrics import accuracy_score
from sklearn.tree import DecisionTreeClassifier

rs = RandomSubspaceDet(estimator=DecisionTreeClassifier(random_state=1))
rs.fit(X_train, y_train)
assert round(accuracy_score(rs.predict(X_test), y_test), 4) == 0.7446

**3 (4 μονάδες)** Υλοποιήστε τη μέθοδο AdaBoost όπως παρουσιάστηκε στο μάθημα. Χρησιμοποιήστε τη συνάρτηση clone από το sklearn.base για να δημιουργήστε αντίγραφο του βασικού μοντέλου σε κάθε επανάληψη. Χρησιμοποιήστε την παράμετρο sample_weight της fit του βασικού μοντέλου για να ορίσετε τα βάρη των παραδειγμάτων εκπαίδευσης. 

In [88]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.base import clone

class AdaBoost:
    def __init__(self, n_estimators=20, estimator=DecisionTreeClassifier(max_depth=1)):
        """
        The function __init__ is the initialization function of the class AdaBoost
        
        ** Note that the class AdaBoost refers to the AdaBoost.M1 method for classification
        problems.
        
        """

        # Initialize the initial estimator model and the number of learning
        # models/estimators that will comprise the model set, constructed by the AdaBoost.M1
        # method. Also, initialize the model set to an empty set. Moreover, initialize
        # the weighted errors that ach estimator of the model set achieves on the
        # training data.
        self.initial_estimator = estimator
        self.n_estimators = n_estimators
        self.estimators_set = []
        self.weighted_errors = np.zeros(self.n_estimators)

    def fit(self, X_train, y_train):
        """
        The fit function is the implementation of the fitting procedure of the models/estimators
        produced by AdaBoost.M1 method for classification problems.
        This function does not return anything.
        
        """
        estimator_name_base_str = 'estimator_'

        # Initialize the weights of the training examples.
        train_examples_weights = np.ones(X_train.shape[0],) * 1/X_train.shape[0]

        for t in range(self.n_estimators):
            estimator = clone(self.initial_estimator)
            estimator.fit(X_train, y_train, sample_weight=train_examples_weights)

            # Compute the weighted error that the current estimator model achieves while
            # making predictions on the training data.
            self.weighted_errors[t] = np.sum(train_examples_weights * np.where(estimator.predict(X_train)!=y_train, 1, 0))

            if (self.weighted_errors[t]==0 or self.weighted_errors[t]>=0.5):
                break

            # Update the weights of the training examples where the estimators makes correct
            # predictions and then normalize the weights of all training examples.
            train_examples_weights = train_examples_weights * np.where(estimator.predict(X_train)==y_train, (self.weighted_errors[t] / (1-self.weighted_errors[t])), 1)

            train_examples_weights = train_examples_weights / np.sum(train_examples_weights)

            # Update the model set with the new estimator
            self.estimators_set.append((estimator_name_base_str + f"{t+1}", estimator))

    def predict(self, X):
        """
        The predict function makes predictions for the instancees in a data set X by combining the
        predictions of the individual estimators in the model set for those instances in X.

        The function returns the class into which it classifies each instance of data set X. To do
        that, this method calculates the weighted votes for each class in the classification problem,
        for each instance in the set X and finally selects (predicts/classifies), for each such
        instance of X, the class that concentrates the majority of the weighted votes. The weighting
        of the votes is based on the calculated (from the AdaBoost.fit call) weighted errors of each
        estimator. 
        
        """

        # Define a 2D array called individual predictions whose each column stores the
        # predictions, made by the corresponding estimator model in the model set, for 
        # the class of each instance in X.
        # Then calculate the total predictions for the class of each instance in X, considering
        # a "weighted voting" logic across all the individual predictions given by estimators
        # in model set.
        individual_predictions = np.array([estimator.predict(X) for (_, estimator) in self.estimators_set]).T

        # Check if the predictions of the individual estimators are the names of the classes
        # for each instance in X or not. In the first case, try to exploit the classes_ attribute
        # of the estimators in model set. Also, in this case, the final predictions must be
        # the name of the class for each instance in X.
        if(type(individual_predictions[0][0]) == str and hasattr(self.estimators_set[0][1], 'classes_')):
            classes = self.estimators_set[0][1].classes_
        else:
            classes = np.unique(individual_predictions)
        
        weighted_votes = np.zeros((X.shape[0], classes.shape[0]))
        for j in range(X.shape[0]):
            for i in range(len(classes)):
                weighted_votes[j][i] = np.sum([
                        (individual_predictions[j,t]==classes[i]) * np.log((1-self.weighted_errors[t])/self.weighted_errors[t]) 
                        for t in range(individual_predictions.shape[1])
                    ])
        
        major_weighted_classes = classes[np.argmax(weighted_votes, axis=1)]

        return major_weighted_classes

In [89]:
"""Τεστ ορθής υλοποίησης AdaBoost"""
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

ab = AdaBoost(n_estimators=20, estimator=DecisionTreeClassifier(max_depth=2, random_state=1))
ab.fit(X_train, y_train)
assert round(accuracy_score(ab.predict(X_test), y_test), 4) == 0.7706


In [90]:
#Ίδιο αποτέλεσμα παίρνουμε και με την κλάση της sklearn. Ωστόσο υπάρχουν γενικότερα διαφορές στην υλοποίηση της σε σ΄χέση με αυτήν που είδαμε στη θεωρία, όπως η υποστήριξη παραπάνω από 2 κλάσεων και ο διαφορετικός χειρισμός της τυχαιότητας στην εκπαίδευση των επί μέρους μοντέλων.
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ab = AdaBoostClassifier(n_estimators=20, estimator=DecisionTreeClassifier(max_depth=2, random_state=1))
ab.fit(X_train, y_train)
assert round(accuracy_score(ab.predict(X_test), y_test), 4) == 0.7706